<a href="https://colab.research.google.com/github/MarkusThill/techdays26/blob/main/TDConnect4Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install bitbully[gui]

In [ ]:
# Only relevant for Google Colab:
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import output

    output.enable_custom_widget_manager()

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import math
import re
import zipfile

import numpy as np

# If you want a static type check against the Protocol:
# from bitbully.agent_protocols import Connect4Agent
# from bitbully import Board


# =============================================================================
# Models
# =============================================================================

@dataclass(frozen=True, slots=True)
class TupleLUT:
    """One n-tuple LUT (keeps both original + mirrored index sets)."""
    n: int
    m: int
    idxs: np.ndarray        # (n,), original indices (mirror1)
    idxs_m: np.ndarray      # (n,), mirrored indices (mirror2)
    lut: np.ndarray         # (P**n,), float64


@dataclass(frozen=True, slots=True)
class PlayerWeights:
    """Weights file for one 'player to move' (p0 or p1)."""
    t: int
    p: int
    luts: tuple[TupleLUT, ...]


@dataclass(frozen=True, slots=True)
class TwoPlayerWeights:
    """Weights for both players to move: p0 (yellow), p1 (red)."""
    p0: PlayerWeights
    p1: PlayerWeights

    def for_player(self, player: int) -> PlayerWeights:
        if player == 0:
            return self.p0
        if player == 1:
            return self.p1
        raise ValueError("player must be 0 (yellow-to-move) or 1 (red-to-move)")


# =============================================================================
# Parsing helpers (text -> TwoPlayerWeights)
# =============================================================================

@dataclass(slots=True)
class Block:
    text: str | list[str]
    children: list["Block"]


_ALLOWED_CHARS = re.compile(r"^[\s0-9+\-\.eE{}]*$")


def _normalize_text(text: str) -> str:
    text = text.replace("\r", "")
    text = text.replace("\n", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def _validate_text(text: str) -> None:
    if text.count("{") != text.count("}"):
        raise ValueError("Brace count mismatch: unbalanced '{' and '}'.")
    if not _ALLOWED_CHARS.match(text):
        for i, ch in enumerate(text):
            if not re.match(r"[\s0-9+\-\.eE{}]", ch):
                raise ValueError(f"Invalid character {ch!r} (ord={ord(ch)}) at position {i}")
        raise ValueError("Invalid character(s) found.")


def _parse_blocks(text: str) -> Block:
    root = Block(text="", children=[])
    stack: list[Block] = [root]
    buf: list[str] = []

    def flush() -> None:
        if not buf:
            return
        s = "".join(buf)
        buf.clear()
        cur = stack[-1]
        assert isinstance(cur.text, str)
        cur.text += s

    for pos, ch in enumerate(text):
        if ch == "{":
            flush()
            child = Block(text="", children=[])
            stack[-1].children.append(child)
            stack.append(child)
        elif ch == "}":
            flush()
            stack.pop()
            if not stack:
                raise ValueError(f"Unmatched '}}' at character index {pos}")
        else:
            buf.append(ch)

    if len(stack) != 1:
        raise ValueError("Unmatched '{' (file ended while blocks were still open).")

    flush()
    return root


def _split_tokens_inplace(block: Block) -> None:
    if isinstance(block.text, str):
        s = block.text.strip()
        block.text = s.split() if s else []
    for child in block.children:
        _split_tokens_inplace(child)


def _block_to_tuple_lut(block: Block, *, p: int, int_dtype=np.int32) -> TupleLUT:
    if not isinstance(block.text, list) or len(block.text) != 2:
        raise ValueError(f"Expected tuple header tokens [N, M], got: {block.text!r}")
    if len(block.children) != 3:
        raise ValueError(f"Expected 3 children (set, mirror, weights), got {len(block.children)}")

    n = int(block.text[0])
    m = int(block.text[1])

    b_set, b_mirror, b_w = block.children
    if not isinstance(b_set.text, list) or not isinstance(b_mirror.text, list) or not isinstance(b_w.text, list):
        raise ValueError("Child blocks must be token lists. Did you call _split_tokens_inplace()?")

    if len(b_set.text) != n:
        raise ValueError(f"Sample set length mismatch: expected {n}, got {len(b_set.text)}")
    if len(b_mirror.text) != n:
        raise ValueError(f"Mirrored set length mismatch: expected {n}, got {len(b_mirror.text)}")

    expected_w = int(p**n)
    if len(b_w.text) != expected_w:
        raise ValueError(f"LUT length mismatch: expected {expected_w} (= {p}^{n}), got {len(b_w.text)}")

    return TupleLUT(
        n=n,
        m=m,
        idxs=np.asarray([int(x) for x in b_set.text], dtype=int_dtype),
        idxs_m=np.asarray([int(x) for x in b_mirror.text], dtype=int_dtype),
        lut=np.asarray([float(x) for x in b_w.text], dtype=np.float64),
    )


class TDWeightsLoader:
    """Load TD-agent weights from `p0.txt` and `p1.txt` (directory or zip)."""

    def __init__(self, *, int_dtype=np.int32, validate: bool = True, strict_t: bool = True) -> None:
        self._int_dtype = int_dtype
        self._validate = validate
        self._strict_t = strict_t

    def _load_from_text(self, raw: str) -> PlayerWeights:
        text = _normalize_text(raw)
        if self._validate:
            _validate_text(text)

        root = _parse_blocks(text)
        _split_tokens_inplace(root)

        if not root.children:
            raise ValueError("No top-level block found. Expected file to start with '{'.")
        file_block = root.children[0]

        if not isinstance(file_block.text, list) or len(file_block.text) < 2:
            raise ValueError(f"Expected file header tokens [T, P], got: {file_block.text!r}")

        t = int(file_block.text[0])
        p = int(file_block.text[1])

        luts = tuple(_block_to_tuple_lut(b, p=p, int_dtype=self._int_dtype) for b in file_block.children)

        if self._validate and self._strict_t and t != len(luts):
            raise ValueError(f"T mismatch: header T={t}, parsed tuples={len(luts)}")

        return PlayerWeights(t=t, p=p, luts=luts)

    def load_file(self, path: str | Path) -> PlayerWeights:
        raw = Path(path).read_text(encoding="utf-8", errors="strict")
        return self._load_from_text(raw)

    def load_file_from_zip(self, zip_path: str | Path, member: str) -> PlayerWeights:
        with zipfile.ZipFile(zip_path, "r") as zf:
            try:
                data = zf.read(member)
            except KeyError as e:
                raise FileNotFoundError(f"{member!r} not found in zip {str(zip_path)!r}") from e
        raw = data.decode("utf-8", errors="strict")
        return self._load_from_text(raw)

    def load_two_player(self, directory: str | Path, *, p0: str = "p0.txt", p1: str = "p1.txt") -> TwoPlayerWeights:
        d = Path(directory)
        w0 = self.load_file(d / p0)
        w1 = self.load_file(d / p1)
        if self._validate and (w0.p != w1.p):
            raise ValueError(f"P mismatch between p0/p1: {w0.p} vs {w1.p}")
        return TwoPlayerWeights(p0=w0, p1=w1)

    def load_two_player_from_zip(
        self,
        zip_path: str | Path,
        *,
        p0: str = "p0.txt",
        p1: str = "p1.txt",
    ) -> TwoPlayerWeights:
        w0 = self.load_file_from_zip(zip_path, p0)
        w1 = self.load_file_from_zip(zip_path, p1)
        if self._validate and (w0.p != w1.p):
            raise ValueError(f"P mismatch between p0/p1: {w0.p} vs {w1.p}")
        return TwoPlayerWeights(p0=w0, p1=w1)


# =============================================================================
# Board encoding (7 columns x 6 rows, bottom->top) -> flat 42 state array
# =============================================================================

def board_cols_to_flat_features(board_cols: list[list[int]]) -> np.ndarray:
    """Convert board_cols[col][row] into 42-length state array with P=4 encoding."""
    if len(board_cols) != 7 or any(len(col) != 6 for col in board_cols):
        raise ValueError("Expected board_cols shape (7, 6): 7 columns each with 6 cells.")

    flat = np.zeros(42, dtype=np.int8)

    for col in range(7):
        column = board_cols[col]

        reachable_row: int | None = None
        for row in range(6):
            if column[row] == 0:
                reachable_row = row
                break

        for row in range(6):
            idx = col * 6 + row
            cell = column[row]
            if cell == 1:
                flat[idx] = 1
            elif cell == 2:
                flat[idx] = 2
            else:
                flat[idx] = 3 if (reachable_row is not None and row == reachable_row) else 0

    return flat


# =============================================================================
# TD evaluator
# =============================================================================

def _lut_index_from_states(states: np.ndarray, p: int) -> int:
    powers = p ** np.arange(states.size, dtype=np.int64)
    return int(states.astype(np.int64, copy=False) @ powers)


class TDEvaluator:
    """Compute TD n-tuple value from TwoPlayerWeights."""

    def __init__(self, weights: TwoPlayerWeights) -> None:
        self._weights = weights

    def value(self, *, board_cols: list[list[int]], player_to_move: int) -> float:
        pw = self._weights.for_player(player_to_move)
        flat = board_cols_to_flat_features(board_cols).astype(np.int64, copy=False)

        total = 0.0
        for tup in pw.luts:
            idx1 = _lut_index_from_states(flat[tup.idxs], pw.p)
            idx2 = _lut_index_from_states(flat[tup.idxs_m], pw.p)
            total += float(tup.lut[idx1]) + float(tup.lut[idx2])
        return total

    @staticmethod
    def to_score(raw_value: float) -> int:
        return int(round(100.0 * math.tanh(raw_value)))


# =============================================================================
# Agent that satisfies the Connect4Agent Protocol
# =============================================================================

class TDConnect4Agent:
    """TD n-tuple agent compatible with `Connect4Agent` Protocol.

    Implements:
      - score_all_moves(board) -> dict[int,int]
      - best_move(board) -> int
      - score_move(board, column, first_guess=0) -> int
    """

    def __init__(self, evaluator: TDEvaluator, *, tie_break: str = "center") -> None:
        self._eval = evaluator
        self._tie_break = tie_break

    # ---- Protocol method 1 ----
    def score_all_moves(self, board) -> dict[int, int]:
        """Return {col: score} for all legal moves. Illegal/full columns are excluded."""
        player_to_move = board.current_player() - 1  # BitBully: {1,2} -> {0,1}
        if player_to_move not in (0, 1):
            raise ValueError(f"Unexpected current_player(): {board.current_player()}")

        scores: dict[int, int] = {}

        for col in range(7):
            if not board.is_legal_move(col):
                continue

            after = board.play_on_copy(col)
            next_player = after.current_player() - 1
            if next_player not in (0, 1):
                raise ValueError(f"Unexpected current_player() after move: {after.current_player()}")

            raw = self._eval.value(board_cols=after.to_array(), player_to_move=next_player)
            score = self._eval.to_score(raw)

            # normalize to "bigger is better for side-to-move"
            if player_to_move == 1:
                score = -score

            scores[col] = score

        return scores

    # ---- Protocol method 2 ----
    def best_move(self, board) -> int:
        """Return best legal move using BitBully-like tie breaking."""
        scores = self.score_all_moves(board)
        if not scores:
            raise ValueError("No legal moves available.")

        best = max(scores.values())
        candidates = [c for c, v in scores.items() if v == best]

        if len(candidates) == 1:
            return candidates[0]

        if self._tie_break == "center":
            center = 3
            return min(candidates, key=lambda c: (abs(c - center), c))
        if self._tie_break == "left":
            return min(candidates)
        if self._tie_break == "right":
            return max(candidates)

        raise ValueError("tie_break must be one of: 'center', 'left', 'right'")

    # ---- Optional Protocol method ----
    def score_move(self, board, column: int, first_guess: int = 0) -> int:
        """Evaluate a single legal move. `first_guess` is accepted for Protocol compatibility."""
        _ = first_guess  # this TD agent ignores it
        if not board.is_legal_move(column):
            raise ValueError(f"Illegal move: column {column}")

        player_to_move = board.current_player() - 1
        after = board.play_on_copy(column)
        next_player = after.current_player() - 1

        raw = self._eval.value(board_cols=after.to_array(), player_to_move=next_player)
        score = self._eval.to_score(raw)

        if player_to_move == 1:
            score = -score

        return score


In [ ]:
# =============================================================================
# Example usage (including Protocol check)
# =============================================================================
loader = TDWeightsLoader(int_dtype=np.int32, validate=True, strict_t=True)

# Directory:
# both = loader.load_two_player(".")

# Zip:
both = loader.load_two_player_from_zip("C4.TCL-EXP-lam0.5-et.txt.zip", p0="p0.txt", p1="p1.txt")

evaluator = TDEvaluator(both)
td_agent = TDConnect4Agent(evaluator, tie_break="center")

import bitbully

board = bitbully.Board()
board.play("3334101133114410")

# Structural/runtime Protocol check (optional):
# from bitbully.agent_protocols import Connect4Agent
# assert isinstance(agent, Connect4Agent)

print(td_agent.score_all_moves(board))
print("best:", td_agent.best_move(board))
print("score col 3:", td_agent.score_move(board, 3))


In [ ]:
bitbully_agent = BitBully(opening_book="12-ply-dist", tie_break="random")

In [ ]:
from bitbully import BitBully
from bitbully.agent_interface import Connect4Agent

agents: dict[str, Connect4Agent] = {
    "BitBully-12ply": bitbully_agent,
    "TD-Agent": td_agent,
}

In [ ]:
from bitbully.gui_c4 import GuiC4

%matplotlib ipympl

c4gui = GuiC4(agents=agents, autoplay=True)

# Display everything
display(c4gui.get_widget())

In [ ]:
from __future__ import annotations

from typing import Protocol

# from bitbully import Board  # in your codebase, use the real import

# TODO: Remove...
class Connect4Agent(Protocol):
    def score_all_moves(self, board) -> dict[int, int]: ...
    def best_move(self, board) -> int: ...
    def score_move(self, board, column: int, first_guess: int = 0) -> int: ...


def play_match(
    agent_yellow: Connect4Agent,
    agent_red: Connect4Agent,
    *,
    start: bitbully.Board | None = None,
    max_plies: int = 42,
    verbose: int = 0,
) -> int:
    """
    Play a full game between two agents on your BitBully `Board`.

    Returns:
        0 -> draw
        1 -> yellow win
        2 -> red win
    """
    board = start.copy() if start is not None else bitbully.Board()

    # Safety: if someone passes a terminal board
    if board.is_game_over():
        w = board.winner()
        return 0 if w is None else int(w)

    plies = 0
    while not board.is_game_over():
        if plies >= max_plies:
            # Should never happen in Connect-4, but keeps us robust
            return 0

        player = board.current_player()  # 1=yellow, 2=red
        agent = agent_yellow if player == 1 else agent_red

        move = agent.best_move(board)

        if not board.is_legal_move(move):
            raise ValueError(
                f"Agent selected illegal move {move} for player {player}. "
                f"Legal moves: {board.legal_moves()}"
            )

        ok = board.play(move)
        if not ok:
            # `play()` returns False on illegal moves; we already checked, so this would indicate a bug.
            raise RuntimeError(f"Board.play({move}) returned False although move was legal.")

        plies += 1

        if verbose >= 1:
            print(board)

    w = board.winner()
    return 0 if w is None else int(w)

In [ ]:
import bitbully

result = play_match(td_agent, bitbully_agent, verbose=1)
# 0 draw, 1 yellow win, 2 red win
print(result)

In [ ]:
from collections import defaultdict

n_matches = 100
total_scores = defaultdict(int)
for _ in range(n_matches):
    result = play_match(td_agent, bitbully_agent, verbose=0)
    total_scores[result] += 1

In [ ]:
total_scores